# Bluebikes Top Stations, Weekday vs Weekend

For a given `YYYY-MM` month (defaults to the latest complete calendar month),
ranks stations by average daily ridership separately for weekdays and
weekends, and reports the top 10 in each. Produces a Markdown report with
both top-10 tables and a Folium map of the union of the two top-10 lists,
colored by whether a station ranks top 10 on weekdays only, weekends only,
or both (gold). Every marker's popup shows the station's weekday and
weekend ridership and rank, regardless of category.

Pulls monthly trip-history data from the [Bluebikes/Hubway System Data bucket](https://s3.amazonaws.com/hubway-data/index.html). Files are named `{YYYYMM}-bluebikes-tripdata.zip` and are available from `201805` onward.

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile

import folium
import numpy as np
import pandas as pd
import requests
import yaml

from shared.plots.style import PALETTE

## Params

In [ ]:
MONTH = None  # YYYY-MM, or None for the latest complete calendar month

## Load or fetch trip data

Cached independently at `data/raw/bluebikes_tripdata/{yyyymm}/data.csv`, so
re-running the notebook only fetches the month if not already on disk.

In [ ]:
BASE_URL = "https://s3.amazonaws.com/hubway-data"


def latest_complete_month():
    first_of_this_month = pd.Timestamp.today().normalize().replace(day=1)
    return (first_of_this_month - pd.DateOffset(months=1)).strftime("%Y-%m")


def fetch_month_tripdata(yyyymm):
    raw_path = Path(f"data/raw/bluebikes_tripdata/{yyyymm}/data.csv")

    if not raw_path.exists():
        # Almost every month is "{yyyymm}-bluebikes-tripdata.zip"; a rare
        # month (e.g. 202511) ships as "{yyyymm}-bluebikes-tripdata.csv.zip" instead.
        for suffix in ("-bluebikes-tripdata.zip", "-bluebikes-tripdata.csv.zip"):
            url = f"{BASE_URL}/{yyyymm}{suffix}"
            response = requests.get(url, timeout=60)
            if response.ok:
                break
        response.raise_for_status()

        with ZipFile(BytesIO(response.content)) as zf:
            csv_name = next(
                name
                for name in zf.namelist()
                if name.endswith(".csv") and "__MACOSX" not in name
            )
            raw_path.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(csv_name) as src, open(raw_path, "wb") as dst:
                dst.write(src.read())

        print(f"Fetched and cached: {raw_path}")
    else:
        print(f"Loaded cached pull: {raw_path}")

    return raw_path


month = MONTH or latest_complete_month()
month_label = pd.Period(month, freq="M").strftime("%B %Y")

month_path = fetch_month_tripdata(pd.Period(month, freq="M").strftime("%Y%m"))

trips = pd.read_csv(
    month_path,
    usecols=["started_at", "start_station_name", "start_lat", "start_lng"],
)
trips.head()

## Per-station stats, split by weekday / weekend

"Days used" is the count of distinct calendar days (within that day type)
with at least one trip starting at the station; "daily ridership" is total
trips at the station on that day type divided by days used; rank is by
daily ridership among all stations active on that day type.

In [ ]:
trips["trip_date"] = trips["started_at"].str.slice(0, 10)
trips["day_type"] = np.where(
    pd.to_datetime(trips["trip_date"]).dt.dayofweek >= 5, "weekend", "weekday"
)

station_day_stats = (
    trips.groupby(["start_station_name", "day_type"])
    .agg(
        total_trips=("trip_date", "size"),
        days_used=("trip_date", "nunique"),
        lat=("start_lat", "median"),
        lng=("start_lng", "median"),
    )
    .reset_index()
)
station_day_stats["daily_ridership"] = (
    station_day_stats["total_trips"] / station_day_stats["days_used"]
)
station_day_stats["rank"] = (
    station_day_stats.groupby("day_type")["daily_ridership"]
    .rank(ascending=False, method="min")
    .astype(int)
)

station_counts = station_day_stats.groupby("day_type")["start_station_name"].nunique()
weekday_station_count = int(station_counts.get("weekday", 0))
weekend_station_count = int(station_counts.get("weekend", 0))


def day_type_table(day_type):
    return (
        station_day_stats[station_day_stats["day_type"] == day_type]
        .set_index("start_station_name")[
            ["total_trips", "days_used", "daily_ridership", "rank", "lat", "lng"]
        ]
        .add_prefix(f"{day_type}_")
    )


weekday_stats = day_type_table("weekday")
weekend_stats = day_type_table("weekend")

station_stats = weekday_stats.join(weekend_stats, how="outer")
station_stats["lat"] = station_stats[["weekday_lat", "weekend_lat"]].mean(axis=1)
station_stats["lng"] = station_stats[["weekday_lng", "weekend_lng"]].mean(axis=1)
station_stats = station_stats.drop(
    columns=["weekday_lat", "weekday_lng", "weekend_lat", "weekend_lng"]
)
station_stats = station_stats.reset_index().rename(
    columns={"start_station_name": "station"}
)
station_stats.head()

## Top 10 weekday and weekend stations

In [ ]:
top_weekday = (
    station_stats.dropna(subset=["weekday_rank"]).sort_values("weekday_rank").head(10)
)
top_weekend = (
    station_stats.dropna(subset=["weekend_rank"]).sort_values("weekend_rank").head(10)
)

overlap_stations = set(top_weekday["station"]) & set(top_weekend["station"])
print(f"{len(overlap_stations)} station(s) rank top 10 on both weekdays and weekends")
top_weekday[["station", "weekday_rank", "weekday_daily_ridership", "weekday_total_trips"]]

## Markdown report

In [ ]:
output_dir = Path(f"outputs/{month}")
output_dir.mkdir(parents=True, exist_ok=True)


def format_top_table(df, day_type):
    lines = [
        "| Rank | Station | Daily ridership | Total trips |",
        "|---|---|---|---|",
    ]
    for row in df.itertuples():
        rank = int(getattr(row, f"{day_type}_rank"))
        ridership = getattr(row, f"{day_type}_daily_ridership")
        total_trips = int(getattr(row, f"{day_type}_total_trips"))
        lines.append(
            f"| {rank} | {row.station} | {ridership:.1f} | {total_trips:,} |"
        )
    return "\n".join(lines)


report_lines = [
    f"# Bluebikes Top Stations, {month_label}",
    "",
    (
        f"Top 10 stations by average daily ridership for {month_label}, ranked "
        "separately for weekdays and weekends."
    ),
    "",
    f"## Top 10 weekday stations (of {weekday_station_count:,} active)",
    "",
    format_top_table(top_weekday, "weekday"),
    "",
    f"## Top 10 weekend stations (of {weekend_station_count:,} active)",
    "",
    format_top_table(top_weekend, "weekend"),
    "",
]

if overlap_stations:
    report_lines += [
        "## Overlap",
        "",
        f"{len(overlap_stations)} station(s) rank in both top 10 lists: "
        + ", ".join(sorted(overlap_stations)),
        "",
    ]

report_path = output_dir / "top_stations_report.md"
report_path.write_text("\n".join(report_lines))
print(f"Saved report: {report_path}")
print("\n".join(report_lines))

## Send report to Slack

In [ ]:
from shared.slack import markdown_to_mrkdwn, send_message

send_message("#data-science", markdown_to_mrkdwn(report_path.read_text()))

## Map: top stations, weekday vs weekend

In [ ]:
def load_carto_api_key():
    secrets_path = Path("../../../config_secrets.yml")
    if not secrets_path.exists():
        print("Warning: config_secrets.yml not found; CARTO map tiles may be rate-limited. See config_secrets_template.yml.")
        return None
    secrets = yaml.safe_load(secrets_path.read_text()) or {}
    key = secrets.get("api_keys", {}).get("carto")
    if not key:
        print("Warning: no CARTO api key set in config_secrets.yml; map tiles may be rate-limited.")
    return key


CATEGORY_COLORS = {
    "weekday": PALETTE[0],
    "weekend": PALETTE[1],
    "overlap": "#D4AF37",  # gold
}
CATEGORY_LABELS = {
    "weekday": "Top 10 weekday only",
    "weekend": "Top 10 weekend only",
    "overlap": "Top 10 both",
}


def categorize(station):
    in_weekday = station in set(top_weekday["station"])
    in_weekend = station in set(top_weekend["station"])
    if in_weekday and in_weekend:
        return "overlap"
    return "weekday" if in_weekday else "weekend"


map_stations = sorted(set(top_weekday["station"]) | set(top_weekend["station"]))
map_df = station_stats[station_stats["station"].isin(map_stations)].copy()
map_df["category"] = map_df["station"].map(categorize)

carto_api_key = load_carto_api_key()
tiles = "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png"
if carto_api_key:
    tiles += f"?key={carto_api_key}"

map_center = [map_df["lat"].mean(), map_df["lng"].mean()]
station_map = folium.Map(location=map_center, zoom_start=13, tiles=tiles, attr="CARTO")


def fmt_side(row, day_type, total_stations):
    rank = getattr(row, f"{day_type}_rank")
    if pd.isna(rank):
        return f"{day_type.capitalize()}: no trips recorded"
    ridership = getattr(row, f"{day_type}_daily_ridership")
    total_trips = int(getattr(row, f"{day_type}_total_trips"))
    return (
        f"{day_type.capitalize()}: {ridership:.1f} trips/day "
        f"({total_trips:,} total), rank {int(rank):,} of {total_stations:,}"
    )


for row in map_df.itertuples():
    popup_html = (
        f"<b>{row.station}</b><br>"
        f"{fmt_side(row, 'weekday', weekday_station_count)}<br>"
        f"{fmt_side(row, 'weekend', weekend_station_count)}"
    )
    folium.CircleMarker(
        location=[row.lat, row.lng],
        radius=8,
        color=CATEGORY_COLORS[row.category],
        fill=True,
        fill_color=CATEGORY_COLORS[row.category],
        fill_opacity=0.85,
        weight=2,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row.station,
    ).add_to(station_map)

legend_items = "".join(
    f'<div><span style="display:inline-block;width:10px;height:10px;'
    f'border-radius:50%;background:{CATEGORY_COLORS[cat]};margin-right:6px;"></span>'
    f"{label}</div>"
    for cat, label in CATEGORY_LABELS.items()
)
legend_html = f"""
<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999;
            background: white; padding: 10px 12px; border: 1px solid #ccc;
            border-radius: 4px; font-size: 12px; font-family: sans-serif;">
{legend_items}
</div>
"""
station_map.get_root().html.add_child(folium.Element(legend_html))

map_path = output_dir / "top_stations_map.html"
station_map.save(map_path)
print(f"Saved map: {map_path}")
display(station_map)